# LongBench-E data tour: Qwen3-8B Q/K/V collection

This notebook walks through the artifact tree produced by
`experiments/stage1/collect_query_stats.py` on LongBench-E with Qwen3-8B. Set `DATA_DIR`
and `DATASET` in the setup cell and run top-to-bottom. Dependencies: `torch` and
`huggingface_hub` (`pip install torch huggingface_hub`).

## Contents

1. **Setup** — point the notebook at a data directory; pull from Hugging Face Hub if missing.
2. **The three tasks** (qasper, hotpotqa, passage_retrieval_en) — prompt and answer statistics.
3. **A worked example** per task — prompt preview, gold, model prediction.
4. **Attention math** — pre-RoPE Q, post-RoPE Q/K, V, GQA expansion, and where each lives in the artifacts.
5. **RoPE verification** — reconstruct `q_post` from `q_pre` using a pure-torch RoPE.
6. **A one-head attention readout** on the stored tensors.
7. **Pooled statistics** — per-layer, per-head moments across prefill tokens.

## 1. Setup

Two dataset bundles are published as Hugging Face dataset repos. Both share the same schema
(Qwen3-8B, LongBench-E, <4k prompt tokens, `enable_thinking=False`):

| `DATASET` | examples | size | use for |
| --- | ---: | ---: | --- |
| `'small'` | 3 (one per task) | ~7 GB | notebook walkthrough, schema inspection |
| `'full'`  | 24 (8 per task)  | ~57 GB | statistics, covariance, per-head studies |

**How the setup cell resolves data**

- `DATA_DIR` is a single directory where datasets are downloaded to / read from.
  Default: `./data` (next to this notebook). Change it to any absolute or relative path.
- `DATASET` picks which bundle to load.
- The cell expects the bundle at `DATA_DIR / SPECS[DATASET]['dirname']`. If that path already
  contains `manifest.json`, it is used directly — no network.
- Otherwise `huggingface_hub.snapshot_download` fetches the dataset from
  `SPECS[DATASET]['repo_id']` into that path. The download is resumable; if it is interrupted,
  re-running the cell picks up where it left off.

Replace the placeholder repo ids in `SPECS` with real ones before sharing (see
`experiments/stage1/notebooks/upload_to_hf.py` for the matching uploader).

In [1]:
import json
import math
from collections import defaultdict
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

# ----- Configuration -------------------------------------------------------
DATA_DIR = Path('data')   # where datasets are downloaded to / read from
DATASET  = 'small'         # 'small' (~7 GB, 3 examples) or 'full' (~57 GB, 24 examples)

SPECS = {
    'small': {
        'dirname': 'query_stats_longbench_under4k_small',
        'repo_id': 'azaad/longbench-qkv-qwen3-small',
        'approx_size': '~7 GB',
    },
    'full': {
        'dirname': 'query_stats_longbench_under4k',
        'repo_id': 'azaad/longbench-qkv-qwen3-full',
        'approx_size': '~57 GB',
    },
}
# ---------------------------------------------------------------------------

spec = SPECS[DATASET]
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = DATA_DIR / spec['dirname']

if not (DATA_ROOT / 'manifest.json').exists():
    if spec['repo_id'].startswith('<'):
        raise RuntimeError(
            f'Dataset not found at {DATA_ROOT} and SPECS[{DATASET!r}]["repo_id"] is a placeholder.\n'
            f'Either place the unpacked bundle at {DATA_ROOT}, or set a real HF dataset repo id.'
        )
    print(f"Dataset '{DATASET}' not found at {DATA_ROOT}.")
    print(f"Fetching from https://huggingface.co/datasets/{spec['repo_id']} ({spec['approx_size']})...")
    snapshot_download(
        repo_id=spec['repo_id'],
        repo_type='dataset',
        local_dir=str(DATA_ROOT),
    )
    print(f'Downloaded to {DATA_ROOT}')

manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
print(f"\nUsing dataset '{DATASET}' at {DATA_ROOT}")
print(f"Model:    {manifest['config']['model']}")
print(f"Dataset:  {manifest['dataset']}")
print(f"Configs:  {manifest['configs']}")
print(f"Length filter: [{manifest['config']['min_tokens']}, {manifest['config']['max_tokens']}] tokens")
print(f"Examples: {len(manifest['examples'])}")


Using dataset 'small' at data/query_stats_longbench_under4k_small
Model:    Qwen/Qwen3-8B
Dataset:  longbench-e
Configs:  ['qasper', 'hotpotqa', 'passage_retrieval_en']
Length filter: [0, 4000] tokens
Examples: 3


/vault/amir/efficient-llm/teamily-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. LongBench-E tasks

LongBench-E is a length-bucketed version of LongBench designed to measure how model
performance degrades with context length. Three configs were collected:

| config | what the prompt asks | gold answer shape | kvpress `max_new_tokens` |
| --- | --- | --- | ---: |
| `qasper_e` | free-form QA over a scientific article | short phrase, `yes`/`no`, or `unanswerable` | 148 |
| `hotpotqa_e` | multi-hop QA over Wikipedia passages | named entity or short phrase | 52 |
| `passage_retrieval_en_e` | "which of these 30 paragraphs is the abstract from?" | a single `Paragraph N` label | 52 |

The per-task `max_new_tokens` budgets come from kvpress's vendored dataset columns; we honor them
so generation lengths match kvpress's published leaderboard conventions. The chat template was
applied with `enable_thinking=False` (kvpress default) so the model answers directly rather than
entering Qwen3's reasoning mode and exhausting the budget on a `<think>` trace.

In [2]:
# Per-config prompt/generation length statistics (read straight from the manifest).
# prompt_length is the chat-templated input length; n_generated is the number of
# new tokens the model actually produced before stopping (bounded by the kvpress budget).
by_cfg = defaultdict(list)
for e in manifest['examples']:
    by_cfg[e['config']].append(e)

def fmt_range(values):
    return f'min={min(values):5d}  mean={sum(values)/len(values):7.1f}  max={max(values):5d}'

COL_W = 34  # width of a fmt_range(...) output; keeps headers and data aligned.
print(f"{'config':25s}  {'n':>2s}  {'prompt_length (tok)':<{COL_W}s}  {'n_generated (tok)':<{COL_W}s}  budget")
for cfg, entries in sorted(by_cfg.items()):
    prompt_lens = [e['prompt_length'] for e in entries]
    gen_lens    = [e['n_generated']   for e in entries]
    budgets     = [e['max_new_tokens_used'] for e in entries]
    print(f"{cfg:25s}  {len(entries):>2d}  {fmt_range(prompt_lens)}  {fmt_range(gen_lens)}  {budgets[0]}")

config                      n  prompt_length (tok)                 n_generated (tok)                   budget
hotpotqa_e                  1  min= 3234  mean= 3234.0  max= 3234  min=    6  mean=    6.0  max=    6  52
passage_retrieval_en_e      1  min= 3848  mean= 3848.0  max= 3848  min=    9  mean=    9.0  max=    9  52
qasper_e                    1  min= 3046  mean= 3046.0  max= 3046  min=    9  mean=    9.0  max=    9  148


In [3]:
# Answer-length distribution (characters) per config, read from metadata.answers.
for cfg, entries in sorted(by_cfg.items()):
    ans_chars = [len(e['metadata']['answers'][0]) for e in entries if e['metadata']['answers']]
    pred_chars = [len(e['generated_text']) for e in entries]
    print(f"{cfg:25s}  gold chars: {fmt_range(ans_chars)}   pred chars: {fmt_range(pred_chars)}")

hotpotqa_e                 gold chars: min=   16  mean=   16.0  max=   16   pred chars: min=   16  mean=   16.0  max=   16
passage_retrieval_en_e     gold chars: min=   12  mean=   12.0  max=   12   pred chars: min=   27  mean=   27.0  max=   27
qasper_e                   gold chars: min=   46  mean=   46.0  max=   46   pred chars: min=   46  mean=   46.0  max=   46


### A worked example from each task

Below we print the first example of each config: the user message the prompt was built from,
the gold answer(s), the model's generated text, and the kvpress generation budget. Prompts are
truncated at 600 characters for readability — the full rendered text is in each `.pt` file
under `prompt_text` (after chat templating, typically 14-17 KB).

In [4]:
def preview(text, n=600):
    t = text.replace('\n', ' ')
    return t if len(t) <= n else t[:n] + '  [... truncated]'

for cfg in sorted(by_cfg):
    e = by_cfg[cfg][0]
    art = torch.load(DATA_ROOT / e['file'], map_location='cpu', weights_only=False)
    msg = art['messages'][0]['content']
    print(f"\n=== {cfg}  (row {e['row_index']})")
    print(f'PROMPT: {preview(msg)}')
    print(f"GOLD:   {art['metadata']['answers']}")
    print(f"PRED:   {art['generated_text']!r}")
    print(f"BUDGET: {art['metadata']['max_new_tokens']} new tokens  (used {e['n_generated']})")
    del art


=== hotpotqa_e  (row 42)
PROMPT: Answer the question based on the given passages. Only give me the answer and do not output any other words.  The following are given passages. Passage 1: Journal of Environmental Economics and Management The Journal of Environmental Economics and Management is a peer-reviewed academic journal of environmental economics published six times per year. It was the official journal of the Association of Environmental and Resource Economists until 2014 and publishes theoretical and empirical papers concerned with the linkage between economic systems and environmental and natural resources. When it wa  [... truncated]
GOLD:   ['Washington, D.C.']
PRED:   'Washington, D.C.'
BUDGET: 52 new tokens  (used 6)



=== passage_retrieval_en_e  (row 99)
PROMPT: Here are 30 paragraphs from Wikipedia, along with an abstract. Please determine which paragraph the abstract is from.  Paragraph 1: The 2nd Battalion, Devonshire Regiment was a Regular Army unit that was serving on the island of Malta as part of the 1st Malta Infantry Brigade (redesignated as the 231st Infantry Brigade in April 1943) and was involved in the siege of Malta from June 1940 until November 1942. In July 1943 the battalion, together with the 231st Brigade, fought in the Allied invasion of Sicily, and, briefly, in the Allied invasion of Italy in September. After Italy the brigade was  [... truncated]
GOLD:   ['Paragraph 10']
PRED:   'The answer is: Paragraph 10'
BUDGET: 52 new tokens  (used 9)



=== qasper_e  (row 40)
PROMPT: You are given a scientific article and a question. Answer the question as concisely as you can, using a single phrase or sentence if possible. If the question cannot be answered based on the information in the article, write "unanswerable". If the question is a yes/no question, answer "yes", "no", or "unanswerable". Do not provide any explanation.  Article: Introduction Social media platforms have made the spreading of fake news easier, faster as well as able to reach a wider audience. Social media offer another feature which is the anonymity for the authors, and this opens the door to many su  [... truncated]
GOLD:   ['words embeddings, style, and morality features', 'words embeddings, style, and morality features']
PRED:   'words embeddings, style, and morality features'
BUDGET: 148 new tokens  (used 9)


## 3. Attention: what the stored tensors represent

### The intuition

Think of attention as a **soft lookup** that every token does against every earlier token. At
each layer the model produces three vectors per token per head:

- **Query ($q$) — "what am I looking for?"** Describes what information this token needs.
- **Key ($k$) — "what do I offer?"** Advertises what this token can supply.
- **Value ($v$) — "what would I hand over if picked?"** The content that flows forward.

Keys are for matching; values are for delivering.

### The math, end to end

Let $h_i \in \mathbb{R}^{d_{\text{model}}}$ be the hidden state (residual-stream vector) at
position $i$, with $d_{\text{model}} = 4096$ for Qwen3-8B. For a single head of dimension
$d_h = 128$:

#### Step 1 — Project, then rotate $q$ and $k$ via RoPE

**1a. Project.** Three learned linear maps turn the hidden state into per-head $q$, $k$, $v$:

$$W_Q h_i \in \mathbb{R}^{d_h}, \qquad W_K h_i \in \mathbb{R}^{d_h}, \qquad W_V h_i \in \mathbb{R}^{d_h}$$

where $W_Q, W_K, W_V \in \mathbb{R}^{d_h \times d_{\text{model}}}$. Qwen3 additionally applies
an RMSNorm on top of $W_Q h_i$ and $W_K h_i$ (`q_norm` / `k_norm`); that norm is part of what we
store as `q_pre`.

**1b. Rotate $q$ and $k$ (RoPE).** Raw attention is order-blind: shuffle the tokens and every
score stays the same. To inject position we want to transform $q$ and $k$ so that their dot
product encodes *where* each token sits — **without distorting anything else** (attention logits
should stay on the same scale so softmax doesn't become more or less peaky).

The trick is to **rotate**. For every position $p$ there is a fixed orthogonal (rotation)
matrix $R_p \in \mathbb{R}^{d_h \times d_h}$, and we apply it to both $q$ and $k$:

$$q_i \leftarrow R_i\, W_Q h_i, \qquad k_i \leftarrow R_i\, W_K h_i.$$

Two properties fall out of this choice, and together they are why RoPE works:

1. **Norm-preserving.** $\|R_p x\| = \|x\|$ for any $x$. Attention logits are neither inflated
   nor attenuated — how peaky softmax gets stays unchanged as we move through positions.
2. **Relative position.** Rotations compose — $R_m^{\top} R_n = R_{n-m}$ — so
   $\langle q_m,\, k_n \rangle$ depends only on the **offset** $n - m$, not on the absolute
   positions $m$ and $n$ separately. Attention becomes translation-equivariant: sliding the
   whole sequence left or right doesn't change any score.

RoPE is applied to **$q$ and $k$ only**, not $v$: position belongs in the match, not the
payload. Rotating $v$ would change *what* gets retrieved, not *where* it came from.

Under the hood $R_p$ is block-diagonal with $d_h/2$ independent $2 \times 2$ rotations at
geometrically-spaced frequencies (Qwen3 uses $\text{base} = 10^6$). Low-frequency blocks rotate
slowly and encode coarse long-range position; high-frequency blocks rotate quickly and encode
fine short-range position — so a single head can resolve both "two tokens ago" and "3000 tokens
back." The matrix is never materialized; in code the rotation is the element-wise

$$R_p\, x = x \odot \cos(\theta_p) + \text{rotate\_half}(x) \odot \sin(\theta_p),
\qquad \theta_{p,\,i} = p \cdot \text{base}^{-2i/d_h}, \ i \in [0, d_h/2).$$

For a visual walkthrough, see
[Rotary Position Embeddings, explained](https://www.youtube.com/watch?v=V8r__fXx7tU).

**1c. Putting it together:**

$$\boxed{\;q_i = R_i\, W_Q\, h_i, \qquad k_i = R_i\, W_K\, h_i, \qquad v_i = W_V\, h_i\;}$$

Our per-example artifacts split this pipeline at every meaningful breakpoint:

| artifact key | equals | where in Step 1 |
| --- | --- | --- |
| `q_pre`  | $W_Q h_i$ (after q-norm) | **before** RoPE  |
| `q_post` | $R_i W_Q h_i$            | **after** RoPE  |
| `k_post` | $R_i W_K h_i$            | **after** RoPE (read from the final KV cache) |
| `v`      | $W_V h_i$                | no RoPE |

Pre-RoPE K (i.e. $W_K h_i$) is not separately captured because only post-RoPE K ends up in the
cache and that is what attention actually reads.

#### Step 2 — Score and normalize

For the query at step $t$, score against every cached key and softmax across positions $i \le t$:

$$a_{ti} = \frac{\exp\!\left(q_t^\top k_i / \sqrt{d_h}\right)}{\sum_{j=1}^{t} \exp\!\left(q_t^\top k_j / \sqrt{d_h}\right)}$$

- A large $q_t^\top k_i$ means "this key strongly matches what query $t$ is looking for."
- Dividing by $\sqrt{d_h}$ keeps logits from blowing up with head dimension — otherwise softmax
  saturates and all attention mass collapses onto one token.
- The **causal mask** is implicit in the sum's upper bound $t$: positions $i > t$ are excluded,
  so information cannot leak back from the future. (Flash-attention applies this as an additive
  $-\infty$ on upper-triangular entries before softmax — same result.)

#### Step 3 — Aggregate and project back into the residual stream

The attention output is a weighted sum of values, mapped back to hidden-state dimension by an
output projection $W_o \in \mathbb{R}^{d_{\text{model}} \times d_h}$, and **added** to the
residual:

$$h_t^{\text{out}} = h_t + \sum_{i=1}^{t} a_{ti}\, W_o\, v_i$$

Each cached $(k_i, v_i)$ pair contributes a residual update
$\Delta h_{ti} = a_{ti} W_o v_i$ to the output — the "soft lookup → residual update" view used
by attention-score-driven cache-compression methods (*Expected Attention*,
Jégou et al., 2025, [arXiv:2510.00636](https://arxiv.org/abs/2510.00636)).

### Shapes and GQA

With $L = 36$ layers, $H_q = 32$ query heads, $H_{kv} = 8$ K/V heads, and $d_h = 128$:

- $Q$ has shape `(batch, H_q=32, seq, d_h=128)`
- $K, V$ have shape `(batch, H_kv=8, seq, d_h=128)`

Qwen3 uses **grouped-query attention (GQA)**: 32 query heads share only 8 K/V heads. Each K/V
head is used by $32 / 8 = 4$ query heads, so before the $Q K^\top$ step K and V are repeated 4×
along the head axis to line up with Q. GQA shrinks the KV cache by 4× with negligible quality
loss — one of the main reasons the cache fits in memory for long contexts.

### What each `.pt` file stores

Every `examples/ex_XXX.pt` is a dict with the following tensors. Using `L` = 36 layers,
`H_q` = 32 query heads, `H_kv` = 8 K/V heads, `d_h` = 128, and `S` = captured sequence length:

| key | shape | dtype | meaning |
| --- | --- | --- | --- |
| `q_pre`   | `(L, H_q,  S, d_h)` | fp16 | Q after q-norm, **before** RoPE |
| `q_post`  | `(L, H_q,  S, d_h)` | fp16 | Q **after** RoPE |
| `k_post`  | `(L, H_kv, S, d_h)` | fp16 | K **after** RoPE (read from the final KV cache) |
| `v`       | `(L, H_kv, S, d_h)` | fp16 | V (RoPE not applied to V) |

The sequence axis covers **both prefill and decode tokens**. In the HF cache, the last generated
token's K is not written back (nothing attends to it), so
`captured_length = prompt_length + n_generated - 1`. `S` therefore spans the whole trajectory,
not just the prompt.

Why these shapes?

- `L = 36` — Qwen3-8B has 36 transformer blocks.
- `H_q = 32, H_kv = 8` — grouped-query attention with group size `32 / 8 = 4`.
- `d_h = 128` — per-head dimension (`hidden_size = 4096 = H_q * d_h`).
- `S` — captured tokens across prefill + decode, varies per example.

In [5]:
# Load the first example and print the tensor layout.
first_entry = manifest['examples'][0]
example_path = DATA_ROOT / first_entry['file']
art = torch.load(example_path, map_location='cpu', weights_only=False)
n_layers, n_q_heads, seq, head_dim = art['q_post'].shape
_, n_kv_heads, _, _ = art['k_post'].shape

print(f'File: {example_path.name}')
print(f"  config          = {art['config']}   row_index = {art['row_index']}")
print(f"  prompt_length   = {art['prompt_length']}")
print(f"  total_length    = {art['total_length']}  (= prompt_length + n_generated)")
print(f"  captured_length = {art['captured_length']} (= total_length - 1; HF cache drops the last gen K)")
print()
print(f'  n_layers   = {n_layers}')
print(f'  n_q_heads  = {n_q_heads}')
print(f'  n_kv_heads = {n_kv_heads}    (GQA group size = {n_q_heads // n_kv_heads})')
print(f'  head_dim   = {head_dim}')
print()
for k in ('q_pre', 'q_post', 'k_post', 'v'):
    t = art[k]
    print(f'  {k:8s}  shape={tuple(t.shape)}   dtype={t.dtype}')

File: ex_000.pt
  config          = qasper_e   row_index = 40
  prompt_length   = 3046
  total_length    = 3055  (= prompt_length + n_generated)
  captured_length = 3054 (= total_length - 1; HF cache drops the last gen K)

  n_layers   = 36
  n_q_heads  = 32
  n_kv_heads = 8    (GQA group size = 4)
  head_dim   = 128

  q_pre     shape=(36, 32, 3054, 128)   dtype=torch.float16
  q_post    shape=(36, 32, 3054, 128)   dtype=torch.float16
  k_post    shape=(36, 8, 3054, 128)   dtype=torch.float16
  v         shape=(36, 8, 3054, 128)   dtype=torch.float16


### Verifying `q_post = RoPE(q_pre)`

We reconstruct `q_post` from `q_pre` using a pure-torch RoPE identical to Qwen3's
`apply_rotary_pos_emb`. If the hook and the reconstruction agree, the artifact layout matches
the attention math.

The rotation angles use `base = 1_000_000` (Qwen3's `rope_theta`) and positions
`0, 1, ..., S-1`. No position-id offset is needed because the capture spans the whole sequence
from token 0.

In [6]:
def build_rope_cos_sin(seq_len, head_dim, base=1_000_000.0, dtype=torch.float32):
    '''Return (cos, sin) each with shape (seq_len, head_dim), matching Qwen3's RoPE.'''
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=dtype) / head_dim))
    t = torch.arange(seq_len, dtype=dtype)
    freqs = torch.outer(t, inv_freq)           # (seq_len, head_dim // 2)
    emb = torch.cat([freqs, freqs], dim=-1)     # (seq_len, head_dim)
    return emb.cos(), emb.sin()

def rotate_half(x):
    half = x.shape[-1] // 2
    return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

def apply_rope(x, cos, sin):
    # x: (..., seq, head_dim),  cos/sin: (seq, head_dim)
    return x.float() * cos + rotate_half(x.float()) * sin

cos, sin = build_rope_cos_sin(seq, head_dim)
q_pre  = art['q_pre'][0, 0]     # layer 0, query head 0 -> (seq, head_dim)
q_post = art['q_post'][0, 0]
q_post_recon = apply_rope(q_pre, cos, sin).to(q_post.dtype)

abs_err = (q_post_recon - q_post).abs()
print(f'Max  abs error : {abs_err.max().item():.4e}')
print(f'Mean abs error : {abs_err.mean().item():.4e}')
print(f'|q_post| mean  : {q_post.abs().float().mean().item():.4e}')

Max  abs error : 7.8125e-03
Mean abs error : 1.5783e-04
|q_post| mean  : 8.7517e-01


The residual is at fp16 rounding (~$10^{-3}$), confirming our stored `q_post` equals
`RoPE(q_pre)` position-by-position.

### A one-layer, one-head attention readout

To close the loop, we compute attention output for one (layer, query head) using the stored
tensors. For query head $h_q$, the corresponding K/V head is $h_{kv} = h_q \,/\, (H_q/H_{kv})$
because of GQA. Here we pick layer 0, query head 0 (-> K/V head 0) and look at the final captured
position's attention distribution.

In [7]:
layer, qh = 0, 0
group_size = n_q_heads // n_kv_heads
kvh = qh // group_size

q = art['q_post'][layer, qh].float()     # (S, d_h)
k = art['k_post'][layer, kvh].float()    # (S, d_h)
v = art['v'][layer, kvh].float()         # (S, d_h)

q_last = q[-1:]                                 # (1, d_h)  - last captured query
scores = q_last @ k.T / math.sqrt(head_dim)     # (1, S)
weights = torch.softmax(scores, dim=-1)         # (1, S)
ctx = weights @ v                               # (1, d_h)

top = weights[0].topk(5)
print(f'Top-5 attention weights for the final query  (layer={layer}, q_head={qh}):')
for w, idx in zip(top.values.tolist(), top.indices.tolist()):
    role = 'prompt' if idx < art['prompt_length'] else f"gen+{idx - art['prompt_length']}"
    print(f'  position {idx:5d}  ({role:10s})  weight = {w:.4f}')
print(f'Context-vector L2 norm: {ctx.norm().item():.4f}')

Top-5 attention weights for the final query  (layer=0, q_head=0):
  position  3051  (gen+5     )  weight = 0.0203
  position  3017  (prompt    )  weight = 0.0202
  position  3043  (prompt    )  weight = 0.0198
  position  3045  (prompt    )  weight = 0.0184
  position  3011  (prompt    )  weight = 0.0164
Context-vector L2 norm: 0.1745


## 4. Pooled statistics across prefill tokens

`pooled_stats.pt` accumulates first and second moments of Q/K/V across **prefill tokens only**
(decode tokens excluded). Downstream analyses can use these directly — they already include the
covariance, so no re-scanning of the per-example files is needed.

Each of `q_pre`, `q_post`, `k_post`, `v` is stored as a **3-tuple** `(mean, cov, second_moment)`:

| entry | shape | meaning |
| --- | --- | --- |
| `mean`          | `(L, H, d_h)`          | per-head mean vector $\mu = \mathbb{E}[x]$ |
| `cov`           | `(L, H, d_h, d_h)`     | per-head covariance $\Sigma = \mathbb{E}[x x^\top] - \mu \mu^\top$ |
| `second_moment` | `(L, H, d_h, d_h)`     | per-head $\mathbb{E}[x x^\top]$ (uncentered) |

plus a scalar `pooled_prefill_tokens` giving the total tokens aggregated. The aggregation is over
the **sequence axis only** — each layer/head gets its own moments. `H = 32` for Q, `H = 8` for
K/V (GQA).

In [8]:
pooled = torch.load(DATA_ROOT / 'pooled_stats.pt', map_location='cpu', weights_only=False)
n_tokens = pooled['pooled_prefill_tokens']
print(f'Total prefill tokens pooled: {n_tokens}\n')
for name in ('q_pre', 'q_post', 'k_post', 'v'):
    mean, cov, second_moment = pooled[name]
    print(f'{name:7s}  mean={tuple(mean.shape)}  cov={tuple(cov.shape)}  second_moment={tuple(second_moment.shape)}')

# Spot-check the covariance at one (layer, head) for pre-RoPE Q.
mean, cov, _ = pooled['q_pre']
cov_00 = cov[0, 0]
eigvals = torch.linalg.eigvalsh(cov_00)
p = eigvals.clamp_min(0)
p = p / p.sum().clamp_min(1e-30)
shannon_rank = float(torch.exp(-(p * p.clamp_min(1e-30).log()).sum()).item())
print(f'\nCov(q_pre) at layer 0, head 0:  shape={tuple(cov_00.shape)}')
print(f'  trace               = {cov_00.diag().sum().item():.4f}')
print(f'  smallest eigenvalue = {eigvals.min().item():.4e}')
print(f'  largest eigenvalue  = {eigvals.max().item():.4e}')
print(f'  effective rank (Shannon) = {shannon_rank:.2f}')

Total prefill tokens pooled: 10128

q_pre    mean=(36, 32, 128)  cov=(36, 32, 128, 128)  second_moment=(36, 32, 128, 128)
q_post   mean=(36, 32, 128)  cov=(36, 32, 128, 128)  second_moment=(36, 32, 128, 128)
k_post   mean=(36, 8, 128)  cov=(36, 8, 128, 128)  second_moment=(36, 8, 128, 128)
v        mean=(36, 8, 128)  cov=(36, 8, 128, 128)  second_moment=(36, 8, 128, 128)

Cov(q_pre) at layer 0, head 0:  shape=(128, 128)
  trace               = 48.8988
  smallest eigenvalue = 1.9286e-09
  largest eigenvalue  = 1.0299e+01
  effective rank (Shannon) = 28.03


## 5. Gold answers and scoring

Each artifact carries a `metadata` dict with the fields kvpress's LongBench-E scorer expects
(`task`, `answers`, `length`, `all_classes`, `max_new_tokens`). The `generated_text` string is
the model's raw decode; pair it with `answers` and pass a DataFrame through kvpress's vendored
`SCORER_REGISTRY['longbench-e']` (at
`experiments/stage1/benchmarks/longbench/calculate_metrics.py::calculate_metrics_e`) to reproduce
scores.

Results from the full 24-example bundle (0-4k bucket):

| config | n | F1 / score |
| --- | ---: | ---: |
| `qasper_e` | 8 | 70.62 |
| `hotpotqa_e` | 8 | 62.50 |
| `passage_retrieval_en_e` | 8 | 100.00 |

The 3-example `small` bundle is meant for schema inspection, not statistically meaningful scoring
— use the `full` bundle for benchmark numbers.